In [18]:
import pandas as pd 
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)

# Tema visual
plt.style.use('default')

plt.rcParams.update({
    'figure.figsize': (5, 4),

    'axes.facecolor': '#FFFFFF',
    'figure.facecolor': '#FFFFFF',

    'axes.edgecolor': '#D0D7DE',

    'grid.color': '#E5E7EB',
    'grid.linestyle': '--',
    'grid.alpha': 0.8,

    'font.size': 10,
})

RANDOM_STATE = 42

In [19]:
features = pl.read_parquet("../data/interim/features_orig.parquet")
target = pl.read_parquet("../data/raw/target.parquet")

df =  features.with_columns(target['y'])

print(f'Filas: {df.height:,} | columnas: {df.width}')

Filas: 2,139,643 | columnas: 41


In [20]:
TARGET = "y"

LEAKAGE_VARS = ['grade']

NUM = [ col for col, dtype in df.schema.items() if dtype.is_numeric() and col != 'y']

CAT = [col for col, dtype in df.schema.items() if dtype == pl.String and col != 'grade']

VARS = NUM + CAT

print(f"Variables de originación: {len(VARS)}  (numéricas: {len(NUM)}, categóricas: {len(CAT)})")


Variables de originación: 39  (numéricas: 29, categóricas: 10)


## Train and split

In [21]:
X = df[VARS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: filas: {X_train.height} & columnas: {X_train.width}")
print(f"Train: filas: {X_test.height} & columnas: {X_test.width}")
print(f"\nTrain Balance:{y_train.value_counts(normalize=True)}")
print(f"\nTest Balance:{y_test.value_counts(normalize=True)}")

Train: filas: 1711714 & columnas: 39
Train: filas: 427929 & columnas: 39

Train Balance:shape: (2, 2)
┌─────┬────────────┐
│ y   ┆ proportion │
│ --- ┆ ---        │
│ i64 ┆ f64        │
╞═════╪════════════╡
│ 0   ┆ 0.86927    │
│ 1   ┆ 0.13073    │
└─────┴────────────┘

Test Balance:shape: (2, 2)
┌─────┬────────────┐
│ y   ┆ proportion │
│ --- ┆ ---        │
│ i64 ┆ f64        │
╞═════╪════════════╡
│ 0   ┆ 0.86927    │
│ 1   ┆ 0.13073    │
└─────┴────────────┘


In [22]:
def missing_values(df, columnas):

    n_filas = df.height

    resumen_filas = []

    for col in df[columnas].columns:
        n_missing = df[col].null_count()
        resumen_filas.append({
            'columna' : col,
            'dtype':str(df.schema[col]),
            'n_unicos':df[col].n_unique(),
            'n_missing':n_missing,
            'pct_missing':round(n_missing/n_filas * 100,2)
        })

    resumen = (
        pl
        .DataFrame(resumen_filas)
        .sort('pct_missing', descending = True)
        .filter(pl.col('n_missing') > 0)
        )

    
    return   resumen

In [23]:
resumen_first = missing_values(X_train, VARS)
print(resumen_first)

shape: (12, 5)
┌────────────────────────────────────┬───────┬──────────┬───────────┬─────────────┐
│ columna                            ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---                                ┆ ---   ┆ ---      ┆ ---       ┆ ---         │
│ str                                ┆ str   ┆ i64      ┆ i64       ┆ f64         │
╞════════════════════════════════════╪═══════╪══════════╪═══════════╪═════════════╡
│ mths_since_recent_bankcard_delinq  ┆ Int64 ┆ 174      ┆ 1314766   ┆ 76.81       │
│ mths_since_recent_revol_delinq     ┆ Int64 ┆ 176      ┆ 1147320   ┆ 67.03       │
│ mths_since_last_delinq             ┆ Int64 ┆ 171      ┆ 872931    ┆ 51.0        │
│ mths_since_last_installment_acc_op ┆ Int64 ┆ 389      ┆ 724086    ┆ 42.3        │
│ num_open_trades_in_6mths           ┆ Int64 ┆ 19       ┆ 692516    ┆ 40.46       │
│ num_installment_acc_op_in_12mths   ┆ Int64 ┆ 19       ┆ 692516    ┆ 40.46       │
│ num_installment_acc_op_in_24mths   ┆ Int64 ┆ 30       ┆ 692

## Imputación de valores faltantes 

De acuerdo a nuestros analisis previo (EDA), estas variables `mths_since_last_delinq`, `mths_since_recent_bankcard_delinq`, `mths_since_recent_revol_delinq`. Estas variables no tienen nulo porque faltan datos, si no porque el usuario/prestatario nunca tuvo morosidad, es un indicador de un historial crediticio. Es de tipo Missing At random (MNAR) ya que no depende de otras variables.

Metodología de imputación:

1. Crearemos una variable binaria (indcador flag) (1/0) para los tres variables, estas variables nos indicarán si anterior a la imputación contaban con valores faltantes.

2. Reemplazaremos los valores nulos con un valor "Centinela" que nos distinga al resto de elementos. En este caso se aplicara del maximo de cada variable + 1000. Esto nos indicará que aquel usuario tiene un buen historial crediticio. 

In [24]:
months_since_vars = [
    "mths_since_last_delinq",
    "mths_since_recent_bankcard_delinq",
    "mths_since_recent_revol_delinq"
]

#Calculamos los sentinemas solo en train
sentinels = {}

for col in months_since_vars:
    max_value = X_train.select(pl.col(col).max()).item()
    sentinels[col] = int(max_value) + 1000

sentinels

{'mths_since_last_delinq': 1202,
 'mths_since_recent_bankcard_delinq': 1202,
 'mths_since_recent_revol_delinq': 1202}

In [25]:
def apply_sentinels(df, sentinels, columns):

    expressions = []
    # extend elementos directamente a la lista
    for col in columns:
        expressions.extend([
            pl.col(col)
            .is_null()
            .cast(pl.Int8)
            .alias(f"flag_never_{col}"),

            pl.col(col)
            .fill_null(sentinels[col])
            .alias(col)
        ])
    
    return df.with_columns(expressions)

In [26]:
#Aplicamos las imputaciones tanto en el train como en el test.
X_train = apply_sentinels(X_train, sentinels, months_since_vars)
X_test = apply_sentinels(X_test, sentinels, months_since_vars)

In [27]:
train_missing = missing_values(X_train, VARS)
print(train_missing)

test_missing = missing_values(X_test, VARS)
print(train_missing)

shape: (9, 5)
┌────────────────────────────────────┬───────┬──────────┬───────────┬─────────────┐
│ columna                            ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---                                ┆ ---   ┆ ---      ┆ ---       ┆ ---         │
│ str                                ┆ str   ┆ i64      ┆ i64       ┆ f64         │
╞════════════════════════════════════╪═══════╪══════════╪═══════════╪═════════════╡
│ mths_since_last_installment_acc_op ┆ Int64 ┆ 389      ┆ 724086    ┆ 42.3        │
│ num_open_trades_in_6mths           ┆ Int64 ┆ 19       ┆ 692516    ┆ 40.46       │
│ num_installment_acc_op_in_12mths   ┆ Int64 ┆ 19       ┆ 692516    ┆ 40.46       │
│ num_installment_acc_op_in_24mths   ┆ Int64 ┆ 30       ┆ 692516    ┆ 40.46       │
│ num_rev_trades_op_in_12mths        ┆ Int64 ┆ 29       ┆ 692516    ┆ 40.46       │
│ num_rev_trades_op_in_24mths        ┆ Int64 ┆ 49       ┆ 692516    ┆ 40.46       │
│ max_bal_owed                       ┆ Int64 ┆ 31787    ┆ 6925

- Ahora con las varianes que tienen terminologia inicial `num_` es un contador, la opción más conservadora es la imputación con 0. Ya que no se cuenta con un registro previo o sin actidad reciente antes de la fecha. 

- Por otro lado, las variables `mths_since_last_installment_acc_op`, `max_bal_owed` y `bal_to_cred_lim`, imputaremos con la mediana. 

Ademas, creamos variables flags binarias (0/1) para cada una de estas imputaciones. Finalmente, otra consideración a tener en cuenta probar estas imputaciones iniciales en un modelo inicial, y posteriormente probar cortando los datos > 2016. 

In [28]:
vintage_block_counts = [
    "num_inq", 
    "num_open_trades_in_6mths",
    "num_installment_acc_op_in_12mths",
    "num_installment_acc_op_in_24mths",
    "num_rev_trades_op_in_12mths",
    "num_rev_trades_op_in_24mths"
]

vintage_block_other = [
    "max_bal_owed",
    "bal_to_cred_lim",
    "mths_since_last_installment_acc_op"
    ]


medians_vintage ={
    col: X_train.select(pl.col(col).median()).item()
    for col in vintage_block_other
}

medians_vintage

{'max_bal_owed': 4403.0,
 'bal_to_cred_lim': 58.0,
 'mths_since_last_installment_acc_op': 13.0}

In [29]:
def imput_vintage_zero(df,columns):
    
    expressions = []

    for col in columns:
        expressions.extend([
            pl.col(col)
            .is_null()
            .cast(pl.Int8)
            .alias(f"Flag_missing_{col}"),


        pl.col(col)
        .fill_null(0)
        .alias(col)
        ]),

    return df.with_columns(expressions)

In [30]:
def imput_vintage_median(df,columns,median):

    expressions = []

    for col in columns:
        expressions.extend([
            pl.col(col)
            .is_null()
            .cast(pl.Int8)
            .alias(f"Flag_missing_{col}"),
        
        pl.col(col)
        .fill_null(median[col])
        .alias(col)
        ])

    return df.with_columns(expressions)
    

In [31]:
#Imputacion de variables tipo contadores
X_train = imput_vintage_zero(X_train, vintage_block_counts)
X_test = imput_vintage_zero(X_test, vintage_block_counts)

In [32]:
train_missing = missing_values(X_train, VARS)
print(train_missing)

test_missing = missing_values(X_test, VARS)
print(train_missing)

shape: (3, 5)
┌────────────────────────────────────┬───────┬──────────┬───────────┬─────────────┐
│ columna                            ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---                                ┆ ---   ┆ ---      ┆ ---       ┆ ---         │
│ str                                ┆ str   ┆ i64      ┆ i64       ┆ f64         │
╞════════════════════════════════════╪═══════╪══════════╪═══════════╪═════════════╡
│ mths_since_last_installment_acc_op ┆ Int64 ┆ 389      ┆ 724086    ┆ 42.3        │
│ max_bal_owed                       ┆ Int64 ┆ 31787    ┆ 692516    ┆ 40.46       │
│ bal_to_cred_lim                    ┆ Int64 ┆ 188      ┆ 692609    ┆ 40.46       │
└────────────────────────────────────┴───────┴──────────┴───────────┴─────────────┘
shape: (3, 5)
┌────────────────────────────────────┬───────┬──────────┬───────────┬─────────────┐
│ columna                            ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---                                ┆ ---   ┆ -

In [33]:
#Imputacion de variables tipo contadores
X_train = imput_vintage_median(X_train, vintage_block_other,medians_vintage)
X_test = imput_vintage_median(X_test, vintage_block_other, medians_vintage)

In [34]:
train_missing = missing_values(X_train, VARS)
print(train_missing)

test_missing = missing_values(X_test, VARS)
print(train_missing)

shape: (0, 5)
┌─────────┬───────┬──────────┬───────────┬─────────────┐
│ columna ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---     ┆ ---   ┆ ---      ┆ ---       ┆ ---         │
│ str     ┆ str   ┆ i64      ┆ i64       ┆ f64         │
╞═════════╪═══════╪══════════╪═══════════╪═════════════╡
└─────────┴───────┴──────────┴───────────┴─────────────┘
shape: (0, 5)
┌─────────┬───────┬──────────┬───────────┬─────────────┐
│ columna ┆ dtype ┆ n_unicos ┆ n_missing ┆ pct_missing │
│ ---     ┆ ---   ┆ ---      ┆ ---       ┆ ---         │
│ str     ┆ str   ┆ i64      ┆ i64       ┆ f64         │
╞═════════╪═══════╪══════════╪═══════════╪═════════════╡
└─────────┴───────┴──────────┴───────────┴─────────────┘


## Tratamiento de Outliers 

In [35]:
X_train[NUM].describe()

statistic,funded_amnt,interest_rate,monthly_payment,annual_income,dept_paym_income_ratio,num_30+_delinq_in_2yrs,num_inq_in_6mths,mths_since_last_delinq,num_open_credit_lines,num_derogatory_pub_rec,total_credit_revolving_bal,used_credit_share,tot_num_credit_lines,num_open_trades_in_6mths,num_installment_acc_op_in_12mths,num_installment_acc_op_in_24mths,mths_since_last_installment_acc_op,num_rev_trades_op_in_12mths,num_rev_trades_op_in_24mths,max_bal_owed,bal_to_cred_lim,num_inq,num_inq_in_12mths,mths_since_recent_bankcard_delinq,mths_since_recent_revol_delinq,loan_term_months,issue_date_year,region_code,earliest_cr_line_year
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6,1.711714e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",14783.143022,13.058979,439.257188,78970.022606,18.087483,0.309852,0.587417,629.900922,11.63887,0.199767,16642.06267,50.469322,24.246178,0.561424,0.403608,0.934377,17.703394,0.782225,1.664814,5228.324697,57.437724,0.600122,1.496613,932.367526,817.456642,42.736294,2015.765516,4.643715,1999.395957
"""std""",9031.904858,4.794735,263.274409,122054.060402,8.421449,0.872727,0.894677,583.829943,5.642366,0.576987,22893.167196,24.638847,12.009401,0.999116,0.789051,1.443803,20.129011,1.340126,2.44632,4450.790844,16.107047,1.248059,2.049713,490.836411,548.422242,10.783945,1.792385,3.103368,7.833571
"""min""",500.0,5.31,4.93,1900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.0,2007.0,0.0,1933.0
"""25%""",8000.0,9.49,249.19,47303.0,11.77,0.0,0.0,32.0,8.0,0.0,5985.0,31.7,15.0,0.0,0.0,0.0,11.0,0.0,0.0,3677.0,54.0,0.0,0.0,1202.0,52.0,36.0,2015.0,2.0,1995.0
"""50%""",12250.0,12.62,373.22,65136.0,17.59,0.0,0.0,1202.0,11.0,0.0,11321.0,50.4,22.0,0.0,0.0,0.0,13.0,0.0,1.0,4403.0,58.0,0.0,1.0,1202.0,1202.0,36.0,2016.0,4.0,2001.0
"""75%""",20000.0,15.88,581.58,95000.0,24.03,0.0,1.0,1202.0,14.0,0.0,20197.0,69.4,31.0,1.0,1.0,1.0,15.0,1.0,3.0,5127.0,62.0,1.0,2.0,1202.0,1202.0,60.0,2017.0,8.0,2005.0
"""max""",40000.0,30.99,1719.83,1.1e8,49.96,58.0,33.0,1202.0,101.0,86.0,2.904836e6,366.6,176.0,17.0,25.0,51.0,511.0,28.0,60.0,1.170668e6,239.0,48.0,67.0,1202.0,1202.0,60.0,2018.0,9.0,2015.0
